# Clase 3 — Hooks, Multi-Agente y A2A

En esta clase agregamos control y coordinación a los agentes:

| Sección | Concepto |
|---------|----------|
| **1. Hooks** | Observabilidad (timing) y enforcement (bloqueo por política) |
| **2. Steering** | Confirmación humana antes de acciones irreversibles |
| **3. Multi-agente** | `as_tool()`, Swarm y Graph |
| **4. `invocation_state`** | Contexto por llamada sin contaminar el historial |
| **5. A2A** | Agentes en procesos separados comunicándose por HTTP |

## Setup

In [ ]:
!pip install "strands-agents[a2a]" strands-agents-tools requests python-dotenv
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("DUFFEL_API_KEY"), "Falta DUFFEL_API_KEY en .env"
print("Setup OK")

## Herramientas base

Las mismas tools que en clases anteriores: `search_flights` (Duffel sandbox) y `get_weather` (Open-Meteo).

In [ ]:
import os
import requests
from strands import Agent, tool
from strands.models.bedrock import BedrockModel


DUFFEL_API_KEY = os.getenv("DUFFEL_API_KEY")
DUFFEL_URL = "https://api.duffel.com/air/offer_requests"


@tool
def search_flights(origin: str, destination: str, date: str) -> dict:
    """Search one-way flights via Duffel sandbox.

    Args:
        origin: IATA airport code (3 letters), e.g. EZE, JFK, MIA.
        destination: IATA airport code (3 letters).
        date: departure date in YYYY-MM-DD format.

    Returns:
        dict with the top 5 cheapest offers sorted by price.
    """
    headers = {
        "Authorization": f"Bearer {DUFFEL_API_KEY}",
        "Duffel-Version": "v2",
        "Content-Type": "application/json",
    }
    payload = {
        "data": {
            "slices": [{"origin": origin, "destination": destination, "departure_date": date}],
            "passengers": [{"type": "adult"}],
            "cabin_class": "economy",
        }
    }
    response = requests.post(DUFFEL_URL, headers=headers, json=payload)
    response.raise_for_status()
    offers = response.json()["data"]["offers"]
    top_5 = sorted(offers, key=lambda o: float(o["total_amount"]))[:5]
    return {
        "route": f"{origin} -> {destination}",
        "date": date,
        "offers": [
            {
                "airline": o["owner"]["name"],
                "price": f"{o['total_amount']} {o['total_currency']}",
                "stops": len(o["slices"][0]["segments"]) - 1,
            }
            for o in top_5
        ],
    }


@tool
def get_weather(city: str, target_date: str) -> dict:
    """Get weather forecast for a city on a specific date.

    Args:
        city: city name, e.g. 'Miami', 'Buenos Aires'.
        target_date: date in YYYY-MM-DD format.

    Returns:
        dict with max temperature and precipitation.
    """
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()
    if not geo.get("results"):
        return {"error": f"City '{city}' not found"}
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, "longitude": lon,
            "daily": "temperature_2m_max,precipitation_sum",
            "start_date": target_date, "end_date": target_date,
        },
    ).json()
    return {
        "city": city,
        "date": target_date,
        "max_temp_c": weather["daily"]["temperature_2m_max"][0],
        "precipitation_mm": weather["daily"]["precipitation_sum"][0],
    }


print("Tools cargadas: search_flights, get_weather")

---

## 1. Hooks

Los hooks son callbacks que se ejecutan antes y después de cada tool call.
Strands los expone como clases que implementan `HookProvider`.

### 1.1 Observabilidad

`ObservabilityHook` registra el nombre de la tool y el tiempo que tarda en ejecutarse.

In [ ]:
import time
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import  (
    BeforeToolCallEvent,
    AfterToolCallEvent,
)

class ObservabilityHook(HookProvider):
    def __init__(self):
        self._start_time: float = 0.0

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.before)
        registry.add_callback(AfterToolCallEvent, self.after)

    def before(self, event: BeforeToolCallEvent) -> None:
        self._start_time = time.time()
        tool_name = event.tool_use['name']
        print(f"\n>>> HOOK: tool.start | {tool_name}")

    def after(self, event: AfterToolCallEvent) -> None:
        elapsed = time.time() - self._start_time
        tool_name = event.tool_use['name']
        print(f">>> HOOK: tool.end   | {tool_name} | {elapsed:.2f}s")


In [ ]:
agent_observed = Agent(
    tools=[search_flights, get_weather],
    hooks=[ObservabilityHook()],
    system_prompt="Sos un asistente de viajes corporativo. Responde en español."
)

response = agent_observed("Buscame vuelos de Buenos aires a Miami el 5 de junio de 2026")

### 1.2 Enforcement — bloquear acciones por política

`EnforcementHook` inspecciona los parámetros de la tool *antes* de que se ejecute
y puede cancelarla seteando `event.cancel_tool`.

In [ ]:
RESTRICTED_DESTINATIONS = {"SFO"}


class EnforcementHook(HookProvider):
    """Blocks flights to restricted destinations before the tool executes."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.gate)

    def gate(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use['name'] == "search_flights":
            dest = event.tool_use.get('input', {}).get("destination", "")
            if dest in RESTRICTED_DESTINATIONS:
                print(f">>> HOOK: BLOCKED | destino {dest} restringido por politica corporativa")
                event.cancel_tool = (
                    f"Destino {dest} restringido por politica corporativa. "
                    f"No se puede buscar vuelos a este destino."
                )


In [ ]:
agent_enforced = Agent(
    tools=[search_flights, get_weather],
    hooks=[ObservabilityHook(),EnforcementHook()],
    system_prompt="Sos un asistente de viajes corporativo. Responde en español."
)

response = agent_enforced("Buscame vuelos a Miami desde Ezeiza el 5 de junio de 2026")



In [ ]:
agent_enforced("Buscame vuelos a San Francisco en la misma fecha")

---

## 2. Steering — confirmación humana antes de acciones irreversibles

`LLMSteeringHandler` inyecta un system prompt que obliga al agente a pedir confirmación
explícita antes de ejecutar acciones irreversibles como `book_flight`.

In [ ]:
from strands.vended_plugins.steering.handlers.llm.llm_handler import LLMSteeringHandler

@tool
def book_flight(airline: str, price: str, origin: str, destination: str) -> dict:
    """Book a flight. This action is irreversible — it charges the corporate card immediately.

    Args:
        airline: airline name, e.g. 'American Airlines'.
        price: total price with currency, e.g. '267.08 USD'.
        origin: IATA origin airport code.
        destination: IATA destination airport code.

    Returns:
        booking confirmation with booking_id.
    """
    return {
        "status": "booked",
        "booking_id": f"ACME-{origin}{destination}-001",
        "airline": airline,
        "price": price,
        "message": "Reserva confirmada. Se cargo a la tarjeta corporativa.",
    }

confirm_steering = LLMSteeringHandler(
    system_prompt=(
        "Antes de ejecutar cualquier accion irreversible como book_flight, "
        "siempre pedile confirmacion explicita al usuario. "
        "Mostrale los detalles del vuelo (aerolinea, precio, ruta) "
        "y espera un 'si' o 'confirmo' antes de proceder."
    )
)

agent_with_steering = Agent(
    tools=[search_flights, book_flight],
    hooks=[ObservabilityHook()],
    plugins=[confirm_steering],
    system_prompt="Sos un asistente de viajes corporativo. Responde en español."
)

agent_with_steering("Reservame el vuelo mas barato de Ezeiza a miami el 5 de junio de 2026, cuando lo encuentres procede con la reserva")

---

## 3. Multi-agente

Strands ofrece tres formas de coordinar agentes: como tool, en Swarm y en Graph.

### 3.1 Agente como tool (`as_tool`)

El método `.as_tool()` convierte un agente en una herramienta que otro agente puede invocar.
El agente padre decide cuándo y cómo usarlo.

In [ ]:
weather_agent = Agent(
    name="weather_agent",
    tools=[get_weather],
    system_prompt="Sos un experto en clima, cuando te pregunten por el clima de una ciudad en una fecha especifica, usa get_weather para consultar el pronostico. Responde en forma concisa con temperatura maxima y probabilidad de lluvia"
)

In [ ]:
travel_agent = Agent(
    tools=[search_flights,
           weather_agent.as_tool(name="weather_agent",description="Get weather forecast for a city on a specific date")],
hooks=[ObservabilityHook()],
system_prompt="Sos un asistente de viajes corporativo. Buscas vuelos con search_flight y el clima lo delegas en el weather agent"
)

travel_agent("Buscame vuelos a Miami desde Ezeiza el 5 de junio de 2026 y decime como va a estar el clima")



### 3.2 Swarm

Un `Swarm` coordina múltiples agentes. El primero actúa como orquestador
y puede delegar en los demás.

In [ ]:
from strands.multiagent import Swarm

flight_agent = Agent(
    name="flight_agent",
    tools=[search_flights],
    system_prompt="Sos un agente experto en buscar vuelos, usa search_flights para encontrar las mejores opciones"
)

weather_agent_swarm = Agent(
    name="weather_agent_swarm",
    tools=[get_weather],
    system_prompt="Sos un experto en clima. Usa get_weather para consultar pronosticos"
)

summary_agent_swarm = Agent(
    name="summary_agent",
    tools=[],
    system_prompt=(
        "Sos un asistente de viajes corporativo. Tu trabajo es recibir la informacion de los otros agentes y "
        "resumirla en una respuesta clara y util para el usuario. "
        "Recibiras informacion de vuelos y clima, sintetizala y responde en español."
    )
)

swarm = Swarm([summary_agent_swarm,flight_agent,weather_agent_swarm])

print("Swarm configurado con 3 agentes")


swarm("Quiero viajar de EZE a MIA el 5 de junio de 2026. Busquen el mejor vuelo y el clima, y denme un resumen claro.")

### 3.3 Graph — pipeline secuencial

`GraphBuilder` define un grafo de ejecución con nodos (agentes) y aristas (dependencias).
Útil para pipelines con orden fijo.

In [ ]:
from strands.multiagent import GraphBuilder

builder = GraphBuilder()

builder.add_node(flight_agent,"search")
builder.add_node(weather_agent_swarm,"weather")
builder.add_node(summary_agent_swarm,"summary")
builder.add_edge("search","weather")
builder.add_edge("weather","summary")
builder.set_execution_timeout(60)
graph = builder.build()

graph("Necesito viajar de buenos aires a nueva york el 5 de junio de 2026")




---

## 4. `invocation_state` — contexto por llamada

`invocation_state` es un dict que se pasa en cada llamada y queda disponible para los hooks
en `event.invocation_state`. Ideal para propagar metadatos (ID de empleado, presupuesto)
sin contaminar el historial de mensajes.

In [ ]:
class ContextLogHook(HookProvider):
    """Lee employee_id del invocation_state y lo loggea en cada tool call."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.log)

    def log(self, event: BeforeToolCallEvent) -> None:
        emp = event.invocation_state.get("employee_id", "desconocido")
        print(f">>> HOOK: empleado={emp} | tool={event.tool_use['name']}")


shared_state = {
    "employee_id": "emp_001",
    "company": "ACME Corp",
    "budget_usd": 2500,
}

context_agent = Agent(
    tools=[flight_agent, get_weather],
    hooks=[ContextLogHook()],
    system_prompt="Sos un asistente de viajes corporativos. Responde en espanol.",
)

response = context_agent(
    "Buscame vuelos de Buenos Aires a Miami el 06 de junio de 2026",
    invocation_state=shared_state,
)

---

## 5. A2A — Agent-to-Agent

El protocolo **A2A** permite que un agente se comunique con otro corriendo en un proceso separado.

- `A2AServer` expone un agente como endpoint HTTP.
- `A2AAgent` actúa como cliente y lo invoca de forma transparente.

> Antes de ejecutar la celda siguiente, levantá el servidor en otra terminal:
> ```bash
> cd clase-3
> python a2a_weather_server.py
> ```

In [ ]:
from strands.agent.a2a_agent import A2AAgent

remote_weather_agent = A2AAgent(
    endpoint="http://localhost:9010",
    name="weather_agent"
)

result = remote_weather_agent("Como esta el clima en miami hoy 2 de junio de 2026?")
print(str(result))